## Overview

This notebook is intended to:

* Display the number of transformers in a given grid.
* Display the number of transformers in a given grid older than twenty years.
* Display the breakdown of transformers older than twenty years by feeders.

This snippet is designed as follows:

* Define the grid of interest.
* Display the number of transformers in the grid.
* Display the number of transformers older than twenty years in the grid and aggregate them by feeders.
* Plot the results.

Feeders are modelled as having a top transformer that is marked as a producer of energy. This denotes a substation where power shifts from generation and transmission networks to a distribution network.

The insights derived from these results may be helpful in multiple analytics use cases for areas like Asset Management, Grid Planning or Grid Maintenance.

---

## Setup

In [1]:
import getpass
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd
import plotly.express as px

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
# Checks for Google Colab: If detected,
# it bypasses reading credentials from local files and securely draws from Colab Secrets.
try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from contextlib import suppress
    print('☁️ Running in Google Colab. Using Colab Secrets for SQLAPI connection')
    # Look for Colab Secrets, fall back to interactive prompts if missing
    with suppress(SecretNotFoundError): os.environ['EDM_HOST'] = userdata.get('EDM_HOST')
    with suppress(SecretNotFoundError): os.environ['EDM_USER'] = userdata.get('EDM_USER')
    with suppress(SecretNotFoundError): os.environ['EDM_PASSWORD'] = userdata.get('EDM_PASSWORD')
else:
    # If running locally, loads connection parameters from a `.env` file (in this directory or parent directories)
    load_dotenv()

# Prompt the user to manually enter any missing connection parameters not found in `.env` file / Google Colab Secrets
if 'EDM_HOST' not in os.environ: os.environ['EDM_HOST'] = input('EDM server address: ').strip()
if 'EDM_USER' not in os.environ: os.environ['EDM_USER'] = input('EDM username: ')

# psycopg2/libpq automatically read the PG* environment variables, so map the shared EDM_* names onto them.
# (The password is only mapped if provided via `.env`/Colab; otherwise libpq falls back to `~/.pgpass`.)
os.environ['PGHOST'] = os.environ['EDM_HOST']
os.environ['PGUSER'] = os.environ['EDM_USER']
os.environ.setdefault('PGDATABASE', 'edm')
if 'EDM_PASSWORD' in os.environ: os.environ['PGPASSWORD'] = os.environ['EDM_PASSWORD']

print('Verifying SQLAPI connection parameters/credentials with connection attempt')

try:
    # Test database connection
    conn = psycopg2.connect('')
    conn.close()
    print('✅ SQLAPI connection parameters/credentials verified')
except psycopg2.OperationalError:
    print('⚠️ SQLAPI credentials check failed (assume missing/incorrect password, but double-check `.env` / secrets!)')
    # Prompt user securely for password
    os.environ['PGPASSWORD'] = getpass.getpass('Enter EDM Password manually: ')

# Keeps `%sql` result tables rendering correctly on newer versions of prettytable
import prettytable
if 'DEFAULT' not in vars(prettytable): prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

# Load the SQL extension and connect
get_ipython().run_line_magic('reload_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://')

Verifying SQLAPI connection parameters/credentials with connection attempt


✅ SQLAPI connection parameters/credentials verified


---

## Transformer Age

#### Input Parameters
Enter the grid ID of interest.

In [3]:
# PARAMETERS (Cell Tag: parameters)
# Do not split this cell structure. Papermill injects automated
# runtime overrides immediately below this block.
# If you don't want to be asked for input for these parameters, you can set them here.
grid_id = None  # Target grid identifier (e.g., 'awefice')

In [4]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
grid_id = grid_id or input("Enter grid ID: ").strip()  # awefice

#### Transformer Count per Grid
Fetch the number of transformers in the entire grid.

In [5]:
%%sql

SELECT COUNT(grid_element_id) as number_of_transformers
FROM grid_element
WHERE grid_id = :grid_id
    AND type = 'Transformer';

 * postgresql://


1 rows affected.


number_of_transformers
10


Fetch the number of transformers that are older than 20 years.

In [6]:
%%sql
SELECT COUNT(grid_element_id) as number_of_transformers
FROM grid_element
WHERE grid_id = :grid_id
    AND type = 'Transformer'
    AND (meta ->> 'commission_date')::timestamp < (SELECT current_date  - interval '20 year');

 * postgresql://


1 rows affected.


number_of_transformers
8


Fetch the transformers older than 20 years, their commission date, and their top feeders. Top feeders are obtained by getting the transformers' main power source (the producer on the path obtained by tracing to the highest_source of power) using the `grid_get_sources` function.

First, use a single query snippet without saving the results.

In [7]:
%%sql

SELECT ge.grid_element_id as transformer,
    ge.meta ->> 'commission_date' as commission_date,
    ggd.grid_element_id as feeder_top_element
FROM grid_element ge
LEFT JOIN grid_get_sources(:grid_id, ge.grid_element_id , 'true') ggd
ON true
WHERE ge.type = 'Transformer'
    AND ggd.is_producer = 'True'
    AND (ge.meta ->> 'commission_date')::timestamp < (SELECT current_date - interval '20 year');

 * postgresql://


8 rows affected.


transformer,commission_date,feeder_top_element
transformer_16,1989-01-01,transformer_2
transformer_2,1989-01-01,transformer_2
transformer_26,1969-01-01,transformer_2
transformer_31,1979-01-01,transformer_2
transformer_36,1989-01-01,transformer_2
transformer_6,1999-01-01,transformer_6
transformer_63,1985-01-01,transformer_6
transformer_92,1995-01-01,transformer_6


Second, aggregate the number of transformers per feeder. Save the query results to a dataframe for visualization purposes.

In [8]:
%%sql feeders <<

SELECT ggd.grid_element_id as feeder_top_element,
    COUNT(ge.grid_element_id) as number_of_transformers
FROM grid_element ge
LEFT JOIN grid_get_sources(:grid_id, ge.grid_element_id , 'true') ggd
ON true
WHERE ge.type = 'Transformer'
    AND ggd.is_producer = 'True'
    AND (ge.meta ->> 'commission_date')::timestamp < (SELECT current_date - interval '20 year')
GROUP BY ggd.grid_element_id;

 * postgresql://


2 rows affected.
Returning data to local variable feeders


In [9]:
# Convert the results to a data frame and display them.
df_feeder = feeders.DataFrame()
df_feeder

,feeder_top_element,number_of_transformers
0,transformer_2,5
1,transformer_6,3


In [10]:
# Plot the results using a bar chart.
fig = px.bar(
    df_feeder.reset_index(),
    x="feeder_top_element",
    y="number_of_transformers",
    title="<b> Number of Transformers Older than 20 Years by Feeder<b>",
    labels={
        "feeder_top_element": "Top-feeder Elements",
        "number_of_transformers": "Number of Transformers",
    },
)
fig.show()

---